In [1]:
import pandas as pd
import os

# Define exact file paths
input_path = r"D:\Data Analytics\CV Projects\Projects\Q-Commerce Logistics & Profitability Command Center\Data files\Delivery_Logistics.csv"
output_dir = r"D:\Data Analytics\CV Projects\Projects\Q-Commerce Logistics & Profitability Command Center\Data files\Updated"
output_csv = os.path.join(output_dir, "Cleaned_Delivery_Logistics_1M.csv")

def run_etl_pipeline():
    df = pd.read_csv(input_path)

    df_1m = pd.concat([df] * 42, ignore_index=True)

    df_1m['delivery_id'] = range(1, len(df_1m) + 1)

    df_1m['delivery_time_hours'] = df_1m['delivery_time_hours'].str.extract(r'0+(\d+)$').astype(float)
    df_1m['expected_time_hours'] = df_1m['expected_time_hours'].str.extract(r'0+(\d+)$').astype(float)

    cols_to_standardize = [
        'delivery_partner', 'package_type', 'vehicle_type',
        'delivery_mode', 'region', 'weather_condition',
        'delivery_status', 'delayed'
    ]
    for col in cols_to_standardize:
        df_1m[col] = df_1m[col].str.title()

    os.makedirs(output_dir, exist_ok=True)
    df_1m.to_csv(output_csv, index=False)

    print("-" * 50)
    print(f"Part 1 Complete! {len(df_1m):,} rows saved successfully at:\n{output_csv}")
    print("-" * 50)

if __name__ == "__main__":
    run_etl_pipeline()

Exporting 1M+ row dataset...
--------------------------------------------------
Part 1 Complete! 1,050,000 rows saved successfully at:
D:\Data Analytics\CV Projects\Projects\Q-Commerce Logistics & Profitability Command Center\Data files\Updated\Cleaned_Delivery_Logistics_1M.csv
--------------------------------------------------


In [2]:
import pandas as pd
import sqlite3
import os
import time

# Define exact file paths
input_csv = r"D:\Data Analytics\CV Projects\Projects\Q-Commerce Logistics & Profitability Command Center\Data files\Updated\Cleaned_Delivery_Logistics_1M.csv"
output_dir = r"D:\Data Analytics\CV Projects\Projects\Q-Commerce Logistics & Profitability Command Center\Data files\Updated"
db_path = os.path.join(output_dir, "Logistics_Command_Center.db")

def build_star_schema():
    start_time = time.time()
    df = pd.read_csv(input_csv)

    # Explicitly defining the 6 required dimensions to prevent naming collisions
    dimensions = [
        ('delivery_partner', 'partner_id', 'Dim_Partner'),
        ('vehicle_type', 'vehicle_id', 'Dim_Vehicle'),
        ('delivery_mode', 'mode_id', 'Dim_Mode'),
        ('region', 'region_id', 'Dim_Region'),
        ('weather_condition', 'weather_id', 'Dim_Weather'),
        ('package_type', 'package_id', 'Dim_Package')
    ]

    print(f"Connecting to SQLite database at {db_path}...")
    conn = sqlite3.connect(db_path)

    # 1. Generate and export the 6 Dimension tables
    for col_name, id_col, table_name in dimensions:
        print(f"Extracting and building {table_name}...")
        # pd.factorize maps unique strings to integers. Adding 1 ensures IDs start at 1, not 0.
        df[id_col], uniques = pd.factorize(df[col_name])
        df[id_col] += 1

        dim_df = pd.DataFrame({
            id_col: range(1, len(uniques) + 1),
            col_name: uniques
        })
        dim_df.to_sql(table_name, conn, if_exists='replace', index=False)

    # 2. Construct and export the Fact table
    fact_columns = [
        'delivery_id', 'partner_id', 'vehicle_id', 'mode_id', 'region_id', 'weather_id', 'package_id',
        'distance_km', 'package_weight_kg', 'delivery_time_hours', 'expected_time_hours',
        'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost'
    ]

    fact_df = df[fact_columns]
    fact_df.to_sql('Fact_Delivery', conn, if_exists='replace', index=False)

    conn.close()

    elapsed = time.time() - start_time
    print("-" * 50)
    print(f"Part 2 Complete! Star Schema Built Successfully in {elapsed:.2f} seconds.")
    print("-" * 50)

if __name__ == "__main__":
    build_star_schema()

Loading 1M+ row dataset into memory...
Connecting to SQLite database at D:\Data Analytics\CV Projects\Projects\Q-Commerce Logistics & Profitability Command Center\Data files\Updated\Logistics_Command_Center.db...
Extracting and building Dim_Partner...
Extracting and building Dim_Vehicle...
Extracting and building Dim_Mode...
Extracting and building Dim_Region...
Extracting and building Dim_Weather...
Extracting and building Dim_Package...
Constructing central Fact_Delivery table...
--------------------------------------------------
Part 2 Complete! Star Schema Built Successfully in 4.09 seconds.
Database contains 1 Fact table and 6 Dimension tables.
--------------------------------------------------
